# NAFNet Image Modification on SageMaker Async Inference

Deploy Megvii's [NAFNet](https://github.com/megvii-research/NAFNet) family behind a single SageMaker **async** endpoint and apply **denoise**, **deblur**, and **stereo super-resolution (NAFSSR)** to an image — or, in the raw editor, to the **masked region only**, non-destructively.

| task | model | what it does |
|------|-------|--------------|
| `denoise` | NAFNet-SIDD-width64 | removes sensor / high-ISO noise |
| `deblur` | NAFNet-GoPro-width64 (`variant="reds"` → NAFNet-REDS-width64) | motion deblur |
| `stereo_sr` | NAFSSR-L_4x | 4× super-resolution from a left+right stereo pair |

**Why one endpoint for three tasks.** All three are NAFNet variants (same blocks, different heads/weights), tiny enough (≤ 67M params) to co-reside on a single 16 GB T4. The handler loads each model lazily on first use and caches it, so a `ml.g4dn.xlarge` never holds more than it needs.

**Layout** (mirrors the hardened `FLUX2-Klein-sagemaker` / `SAM` pattern in this workspace):
- Weights are uncompressed objects under an S3 prefix, mounted verbatim at `/opt/ml/model/`.
- `code/inference.py` + `code/requirements.txt` + `code/nafnet_archs.py` live in a `code/` subfolder inside that prefix — the toolkit auto-discovers `/opt/ml/model/code/inference.py`. Editing the handler = re-upload a few KB.
- The NAFNet architectures are vendored into `nafnet_archs.py`, so the container needs **no** BasicSR install.
- **Async**, not real-time: the image rides through S3 (base64 in JSON), sidestepping the real-time 6 MB / 60 s limits — the exact same contract the editor already uses for SAM 3.

In [ ]:
%pip install -q "sagemaker>=2.254.1" boto3

## 1. Setup

In [ ]:
import boto3
import sagemaker

sess = sagemaker.Session()
region = sess.boto_region_name
bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except Exception:
    role = boto3.client("iam").get_role(RoleName="<your-sagemaker-execution-role>")["Role"]["Arn"]

# --- Configuration -------------------------------------------------------
WEIGHTS_PREFIX = "nafnet/weights"          # S3 prefix that holds weights + code/
ENDPOINT_NAME  = "nafnet-imgmod"            # endpoint AND endpoint-config name

# The user asked for ml.g4dn.xlarge (NVIDIA T4, 16 GB). NAFNet fits with room
# to spare; use tiling (see the invoke helper) for very large SR inputs.
INSTANCE_TYPE  = "ml.g4dn.xlarge"
# PyTorch 2.6 inference DLC (py312, CUDA 12.4) — newest URI the sagemaker<3.0
# SDK resolves. NAFNet uses only stock ops so any recent torch works.
FRAMEWORK_VER  = "2.6"
PY_VER         = "py312"

weights_s3_uri = f"s3://{bucket}/{WEIGHTS_PREFIX}"
print(f"region={region}\nbucket={bucket}\nweights={weights_s3_uri}\nrole={role}")

## 2. Stage weights to S3 (one-time)

`prepare_weights.py` pulls the three NAFNet checkpoints from the community HF mirror `mikestealth/nafnet-models` (no token) and the NAFSSR-L 4× checkpoint from Megvii's official Google Drive (via `gdown`). Skip this cell if the prefix is already populated.

```bash
pip install "huggingface_hub>=0.26" gdown
python prepare_weights.py --out weights
aws s3 sync weights/ s3://<bucket>/nafnet/weights/
```

In [ ]:
import subprocess, sys

# Stage all four checkpoints locally, then sync to S3. Pass --skip-stereo to
# omit the NAFSSR Google Drive download if you only need denoise/deblur.
subprocess.check_call([sys.executable, "prepare_weights.py", "--out", "weights"])
subprocess.check_call(["aws", "s3", "sync", "weights/", f"{weights_s3_uri}/"])
print("Weights synced to", weights_s3_uri)

## 3. Upload the `code/` bundle into the weights prefix

With uncompressed-prefix `model_data` we do **not** pass `entry_point` / `source_dir` (that triggers the SDK repack path, which silently drops a dict `model_data`). The handler lives at `<prefix>/code/inference.py` and the toolkit picks it up automatically. Re-run this cell after any edit to `inference.py` or `nafnet_archs.py`.

In [ ]:
s3 = boto3.client("s3")
for fname in ["inference.py", "requirements.txt", "nafnet_archs.py"]:
    s3.upload_file(f"code/{fname}", bucket, f"{WEIGHTS_PREFIX}/code/{fname}")
    print(f"uploaded code/{fname} -> s3://{bucket}/{WEIGHTS_PREFIX}/code/{fname}")

## 4. Deploy the async endpoint

First deploy takes ~5–8 min (image pull + weight mount). Async because we read the input image from S3 and write the JSON result back to S3.

In [ ]:
from sagemaker.pytorch import PyTorchModel
from sagemaker.async_inference import AsyncInferenceConfig

async_config = AsyncInferenceConfig(
    output_path=f"s3://{bucket}/nafnet-outputs/",
    failure_path=f"s3://{bucket}/nafnet-failures/",
    max_concurrent_invocations_per_instance=2,
)

model_data = {
    "S3DataSource": {
        "S3Uri": f"{weights_s3_uri}/",
        "S3DataType": "S3Prefix",
        "CompressionType": "None",
    }
}

model = PyTorchModel(
    role=role,
    model_data=model_data,
    framework_version=FRAMEWORK_VER,
    py_version=PY_VER,
    env={
        "SAGEMAKER_MODEL_SERVER_TIMEOUT": "3600",
        "TS_DEFAULT_RESPONSE_TIMEOUT": "3600",
        "SAGEMAKER_PROGRAM": "inference.py",
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
    },
)

# Clean slate so we never redeploy on top of a stale model/config.
sm = boto3.client("sagemaker")
for _del in (lambda: sm.delete_endpoint(EndpointName=ENDPOINT_NAME),
             lambda: sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)):
    try:
        _del()
    except sm.exceptions.ClientError:
        pass

predictor = model.deploy(
    endpoint_name=ENDPOINT_NAME,
    endpoint_config_name=ENDPOINT_NAME,
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    async_inference_config=async_config,
    container_startup_health_check_timeout=1800,
)
print("Deployed:", ENDPOINT_NAME)

## 5. Invocation helper

Async flow (identical contract to the editor's SAM 3 call): PUT the request JSON (base64 image[s] + task) to S3, call `invoke_endpoint_async`, poll the returned `OutputLocation` until the result lands.

In [ ]:
import base64, io, json, time, uuid
from urllib.parse import urlparse
from PIL import Image

sm_runtime = boto3.client("sagemaker-runtime")
s3 = boto3.client("s3")

def _b64_file(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("ascii")

def _b64_image(img, fmt="PNG"):
    buf = io.BytesIO(); img.save(buf, format=fmt); return base64.b64encode(buf.getvalue()).decode("ascii")

def _decode_b64(b64):
    return Image.open(io.BytesIO(base64.b64decode(b64)))

def _s3_get(uri):
    p = urlparse(uri); return s3.get_object(Bucket=p.netloc, Key=p.path.lstrip("/"))["Body"].read()

def _s3_exists(uri):
    p = urlparse(uri)
    try:
        s3.head_object(Bucket=p.netloc, Key=p.path.lstrip("/")); return True
    except s3.exceptions.ClientError:
        return False

def modify(payload, timeout=600, poll=3):
    """Submit one image-modification request and return the parsed result dict."""
    in_key = f"nafnet-inputs/req-{uuid.uuid4().hex}.json"
    s3.put_object(Bucket=bucket, Key=in_key, Body=json.dumps(payload).encode("utf-8"))
    resp = sm_runtime.invoke_endpoint_async(
        EndpointName=ENDPOINT_NAME,
        InputLocation=f"s3://{bucket}/{in_key}",
        ContentType="application/json",
    )
    out_uri, fail_uri = resp["OutputLocation"], resp.get("FailureLocation")
    print(f"submitted task={payload['task']}; waiting on {out_uri}")
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _s3_exists(out_uri):
            return json.loads(_s3_get(out_uri))
        if fail_uri and _s3_exists(fail_uri):
            raise RuntimeError(_s3_get(fail_uri).decode("utf-8", "replace"))
        time.sleep(poll)
    raise TimeoutError("Timed out waiting for the result.")

## 6. Test · Denoise

Put a noisy image next to this notebook as `sample.png` (or change the path). Shows the input next to the NAFNet-SIDD denoised output.

In [ ]:
import matplotlib.pyplot as plt

def show_pair(before, after, titles=("input", "output")):
    fig, ax = plt.subplots(1, 2, figsize=(16, 8))
    ax[0].imshow(before); ax[0].set_title(titles[0]); ax[0].axis("off")
    ax[1].imshow(after);  ax[1].set_title(titles[1]); ax[1].axis("off")
    plt.tight_layout(); plt.show()

SRC = "sample.png"
res = modify({"task": "denoise", "image": _b64_file(SRC)})
out = _decode_b64(res["image"])
print("image_size", res["image_size"], "-> output_size", res["output_size"])
show_pair(Image.open(SRC), out, ("noisy input", "NAFNet-SIDD denoised"))

## 7. Test · Deblur (GoPro and REDS variants)

In [ ]:
BLUR = "blurry.png"  # a motion-blurred image next to this notebook

res_gopro = modify({"task": "deblur", "image": _b64_file(BLUR), "variant": "gopro"})
show_pair(Image.open(BLUR), _decode_b64(res_gopro["image"]), ("blurry input", "NAFNet-GoPro deblurred"))

res_reds = modify({"task": "deblur", "image": _b64_file(BLUR), "variant": "reds"})
show_pair(Image.open(BLUR), _decode_b64(res_reds["image"]), ("blurry input", "NAFNet-REDS deblurred"))

## 8. Test · Stereo super-resolution (NAFSSR 4×)

NAFSSR takes a **left + right** low-resolution pair and returns both views at 4×. Provide `left.png` and `right.png` (rectified stereo). Keep inputs small (e.g. ≤ 256×256 each) on a T4 — 4× SR is the most VRAM-hungry task.

In [ ]:
LEFT, RIGHT = "left.png", "right.png"
res_sr = modify({"task": "stereo_sr", "image": _b64_file(LEFT), "image_right": _b64_file(RIGHT)})
print("in", res_sr["image_size"], "-> out", res_sr["output_size"])
show_pair(Image.open(LEFT), _decode_b64(res_sr["image"]), ("left LR", "left 4x SR"))
show_pair(Image.open(RIGHT), _decode_b64(res_sr["image_right"]), ("right LR", "right 4x SR"))

## 9. Test · Masked-region only (the editor use-case)

This mirrors exactly how the raw editor uses the SAM 3 endpoint: you already have a **mask** (from SAM semantic segmentation, or a brush stroke). We send the image **plus the mask**, and the endpoint composites the deblurred result over the original so **only the masked pixels change** — everything outside the mask is byte-for-byte the input. In the SPA this is done client-side and stored as one undoable History step on top of the untouched RAW working buffer.

We run on the **blurry** image (so the effect is visible) and show four panels: the original, the mask drawn on the original, and the masked-region deblur for **both** weight variants (GoPro and REDS). Deblur on an already-sharp `sample.png` would barely change — that's why this cell uses `blurry.png`.

In [ ]:
from PIL import ImageDraw
import numpy as np

def show_row(images, titles):
    n = len(images)
    fig, ax = plt.subplots(1, n, figsize=(7 * n, 7))
    for a, im, t in zip(ax, images, titles):
        a.imshow(im); a.set_title(t); a.axis("off")
    plt.tight_layout(); plt.show()

def mask_overlay(img, mask, color=(255, 0, 0), alpha=0.45):
    """Draw the mask as a translucent colored region on top of img (for display)."""
    base = np.asarray(img.convert("RGB"), np.float32)
    a = (np.asarray(mask.resize(img.size), np.float32) / 255.0)[..., None] * alpha
    tint = np.array(color, np.float32)
    return Image.fromarray((base * (1 - a) + tint * a + 0.5).astype(np.uint8), "RGB")

BLUR = "blurry.png"   # use the blurry image so the masked deblur is clearly visible
img = Image.open(BLUR).convert("RGB")

# Circular mask centered on the image (white = apply effect). In the app this
# is the real mask alpha from SAM / a brush stroke.
mask = Image.new("L", img.size, 0)
d = ImageDraw.Draw(mask)
cx, cy = img.width // 2, img.height // 2
r = min(img.width, img.height) // 3
d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255)

# Generate the masked-region result for BOTH deblur variants.
img_b64, mask_b64 = _b64_image(img), _b64_image(mask)
out_gopro = _decode_b64(modify({"task": "deblur", "variant": "gopro", "image": img_b64, "mask": mask_b64})["image"])
out_reds  = _decode_b64(modify({"task": "deblur", "variant": "reds",  "image": img_b64, "mask": mask_b64})["image"])

# Sanity check: only masked pixels changed (non-destructive).
a = (np.asarray(mask) > 127)
for name, out in [("gopro", out_gopro), ("reds", out_reds)]:
    diff = np.abs(np.asarray(img, np.int16) - np.asarray(out, np.int16)).sum(-1)
    print(f"{name}: max change OUTSIDE mask={int(diff[~a].max())} (expect 0), "
          f"mean change INSIDE mask={diff[a].mean():.2f}")

show_row([img, mask_overlay(img, mask), out_gopro, out_reds],
         ["original (blurry)", "mask region", "deblur GoPro (masked)", "deblur REDS (masked)"])

## 10. Cleanup

The GPU endpoint is the main cost — delete it when idle. Keeping the endpoint **config** lets you re-create ("start") the endpoint later with just CreateEndpoint, the same start/stop pattern the editor's SAM tab uses. The `nafnet/weights/` prefix can stay for fast redeploys; put an S3 lifecycle rule on `nafnet-inputs/` / `nafnet-outputs/` / `nafnet-failures/`.

In [ ]:
# Delete just the endpoint (keeps the config for a fast restart).
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
print("endpoint deleted (config kept for restart)")

# To also remove the config + model, uncomment:
# sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
# sess.delete_model(model.name)